# MNIST Digit Classifier from Scratch using NumPy

## 1. Data Handling

In this section, we'll load the MNIST dataset from the `mnist_train.csv` and `mnist_test.csv` files. We will then prepare the data for training our neural network by performing the following steps:

1.  **Load Data**: We use the `pandas` library to load the CSV files into DataFrames.
2.  **Normalization**: The pixel values of the images range from 0 to 255. We'll scale them to a range of 0 to 1 by dividing by 255. This helps in faster convergence of the model.
3.  **One-Hot Encoding**: The labels are digits from 0 to 9. We need to convert these categorical labels into a binary format. One-hot encoding creates a vector of size 10 for each label, where the element corresponding to the digit is 1 and all other elements are 0.
4.  **Validation Split**: We'll split the training data into a training set and a validation set. The validation set will be used to monitor the model's performance on unseen data during training and to prevent overfitting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    train_df = pd.read_csv('mnist_train.csv')
    test_df = pd.read_csv('mnist_test.csv')
except FileNotFoundError:
    print('mnist_train.csv and/or mnist_test.csv not found.')
    print('Please download the files from https://www.kaggle.com/datasets/oddrationale/mnist-in-csv and place them in the same directory as this notebook.')

In [ ]:
def preprocess_data(df, is_train=True):
    # The first column is the label, the rest are the pixel values
    labels = df.iloc[:, 0].values
    images = df.iloc[:, 1:].values
    
    # Normalize the images
    images = images / 255.0
    
    # One-hot encode the labels
    one_hot_labels = np.zeros((labels.size, 10))
    one_hot_labels[np.arange(labels.size), labels] = 1
    
    if is_train:
        # Split training data into training and validation sets (90% train, 10% validation)
        num_samples = images.shape[0]
        validation_size = int(num_samples * 0.1)
        
        # Shuffle the data
        permutation = np.random.permutation(num_samples)
        images = images[permutation]
        one_hot_labels = one_hot_labels[permutation]
        labels = labels[permutation]

        x_val = images[:validation_size]
        y_val = one_hot_labels[:validation_size]
        y_val_labels = labels[:validation_size]
        
        x_train = images[validation_size:]
        y_train = one_hot_labels[validation_size:]
        
        return x_train, y_train, x_val, y_val, y_val_labels
    else:
        return images, one_hot_labels, labels

if 'train_df' in globals():
  x_train, y_train, x_val, y_val, y_val_labels = preprocess_data(train_df)
  x_test, y_test, y_test_labels = preprocess_data(test_df, is_train=False)

  print(f"Training data shape: {x_train.shape}")
  print(f"Training labels shape: {y_train.shape}")
  print(f"Validation data shape: {x_val.shape}")
  print(f"Validation labels shape: {y_val.shape}")
  print(f"Test data shape: {x_test.shape}")
  print(f"Test labels shape: {y_test.shape}")

## 2. Neural Network Architecture

We will build a simple Artificial Neural Network (ANN) with one hidden layer.

- **Input Layer**: 784 nodes, corresponding to the 28x28 pixels of each MNIST image.
- **Hidden Layer**: 128 nodes with the **ReLU** activation function.
- **Output Layer**: 10 nodes, one for each digit (0-9), with the **Softmax** activation function.

### Weight Initialization

We use **He Initialization** for the weights. This method is well-suited for layers with ReLU activation functions, as it helps prevent the vanishing/exploding gradients problem. The weights are initialized from a normal distribution with mean 0 and a standard deviation of `sqrt(2 / n_in)`, where `n_in` is the number of input units to the layer.

In [ ]:
class NeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size):
        # He initialization for weights
        self.W1 = np.random.randn(hidden_size, input_size) * np.sqrt(2. / input_size)
        self.b1 = np.zeros((hidden_size, 1))
        self.W2 = np.random.randn(output_size, hidden_size) * np.sqrt(2. / hidden_size)
        self.b2 = np.zeros((output_size, 1))

    def relu(self, Z):
        return np.maximum(0, Z)
    
    def relu_derivative(self, Z):
        return Z > 0

    def softmax(self, Z):
        expZ = np.exp(Z - np.max(Z, axis=0, keepdims=True)) # For numerical stability
        return expZ / np.sum(expZ, axis=0, keepdims=True)
    
    def forward(self, X):
        self.Z1 = self.W1.dot(X.T) + self.b1
        self.A1 = self.relu(self.Z1)
        self.Z2 = self.W2.dot(self.A1) + self.b2
        self.A2 = self.softmax(self.Z2)
        return self.A2
    
    def cross_entropy_loss(self, Y_hat, Y):
        m = Y.shape[0]
        loss = - (1/m) * np.sum(Y.T * np.log(Y_hat + 1e-8))
        return loss
    
    def backward(self, X, Y):
        m = X.shape[0]
        dZ2 = self.A2 - Y.T
        self.dW2 = (1/m) * dZ2.dot(self.A1.T)
        self.db2 = (1/m) * np.sum(dZ2, axis=1, keepdims=True)
        dZ1 = self.W2.T.dot(dZ2) * self.relu_derivative(self.Z1)
        self.dW1 = (1/m) * dZ1.dot(X)
        self.db1 = (1/m) * np.sum(dZ1, axis=1, keepdims=True)
        
    def update_parameters(self, learning_rate):
        self.W1 -= learning_rate * self.dW1
        self.b1 -= learning_rate * self.db1
        self.W2 -= learning_rate * self.dW2
        self.b2 -= learning_rate * self.db2
        
    def get_predictions(self, Y_hat):
        return np.argmax(Y_hat, axis=0)
    
    def get_accuracy(self, predictions, Y):
        return np.sum(predictions == np.argmax(Y, axis=1)) / Y.shape[0]
    
    def train(self, X, Y, epochs, learning_rate, batch_size, validation_data):
        losses = []
        accuracies = []
        val_losses = []
        val_accuracies = []
        
        x_val, y_val = validation_data
        
        for epoch in range(epochs):
            # Mini-batch gradient descent
            permutation = np.random.permutation(X.shape[0])
            X_shuffled = X[permutation]
            Y_shuffled = Y[permutation]

            for i in range(0, X.shape[0], batch_size):
                X_batch = X_shuffled[i:i+batch_size]
                Y_batch = Y_shuffled[i:i+batch_size]
                
                Y_hat = self.forward(X_batch)
                self.backward(X_batch, Y_batch)
                self.update_parameters(learning_rate)
            
            # Evaluate on training data
            Y_hat_train = self.forward(X)
            loss = self.cross_entropy_loss(Y_hat_train, Y)
            accuracy = self.get_accuracy(self.get_predictions(Y_hat_train), Y)
            losses.append(loss)
            accuracies.append(accuracy)
            
            # Evaluate on validation data
            Y_hat_val = self.forward(x_val)
            val_loss = self.cross_entropy_loss(Y_hat_val, y_val)
            val_accuracy = self.get_accuracy(self.get_predictions(Y_hat_val), y_val)
            val_losses.append(val_loss)
            val_accuracies.append(val_accuracy)
            
            if (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1}/{epochs} - loss: {loss:.4f} - accuracy: {accuracy:.4f} - val_loss: {val_loss:.4f} - val_accuracy: {val_accuracy:.4f}")
                
        return losses, accuracies, val_losses, val_accuracies

## 3. Forward and Backward Propagation

### Forward Propagation

The input data flows through the network layers to produce an output.

1. **Hidden Layer**:
   - $Z^{[1]} = W^{[1]} X + b^{[1]}$
   - $A^{[1]} = \text{ReLU}(Z^{[1]})$

2. **Output Layer**:
   - $Z^{[2]} = W^{[2]} A^{[1]} + b^{[2]}$
   - $A^{[2]} = \text{Softmax}(Z^{[2]})$

### Backward Propagation

This is where the network learns. We calculate the gradient of the loss function with respect to the network's parameters (weights and biases) and use these gradients to update the parameters. The goal is to minimize the cross-entropy loss $L = -\sum{Y \log(A^{[2]})}$.

A key insight is that the derivative of the Cross-Entropy Loss with respect to the pre-activation values of the output layer ($Z^{[2]}$) simplifies to a very elegant form: $dZ^{[2]} = A^{[2]} - Y$. This combines the derivative of the softmax function and the derivative of the cross-entropy loss, avoiding the need to compute them separately and preventing potential numerical instability issues.

1. **Output Layer Gradients** (Chain Rule: $\frac{\partial L}{\partial W^{[2]}} = \frac{\partial L}{\partial A^{[2]}} \frac{\partial A^{[2]}}{\partial Z^{[2]}} \frac{\partial Z^{[2]}}{\partial W^{[2]}}$):
   - $dZ^{[2]} = A^{[2]} - Y$ (Gradient of the loss w.r.t. the output of the last layer)
   - $dW^{[2]} = \frac{1}{m} dZ^{[2]} (A^{[1]})^T$ (Gradient for the weights of the second layer)
   - $db^{[2]} = \frac{1}{m} \sum dZ^{[2]}$ (Gradient for the biases of the second layer)

2. **Hidden Layer Gradients** (Chain Rule backpropagates the error):
   - $dZ^{[1]} = (W^{[2]})^T dZ^{[2]} * g^{[1]'}(Z^{[1]})$ where $g^{[1]'}$ is the derivative of the ReLU function. This step propagates the error from the output layer back to the hidden layer.
   - $dW^{[1]} = \frac{1}{m} dZ^{[1]} X^T$ (Gradient for the weights of the first layer)
   - $db^{[1]} = \frac{1}{m} \sum dZ^{[1]}$ (Gradient for the biases of the first layer)

## 4. Training the Network

We use **Stochastic Gradient Descent (SGD)** to train the network. Instead of processing the entire dataset at once, we divide it into mini-batches. For each mini-batch, we perform forward propagation, backward propagation, and then update the network's parameters. This approach is computationally efficient and often leads to faster convergence.

The training loop will run for a specified number of epochs, and at the end of each epoch, we will calculate the cross-entropy loss and accuracy on both the training and validation sets to monitor the learning progress.

In [ ]:
if 'train_df' in globals():
    # Network configuration
    INPUT_SIZE = 784
    HIDDEN_SIZE = 128
    OUTPUT_SIZE = 10
    EPOCHS = 100
    LEARNING_RATE = 0.1
    BATCH_SIZE = 64

    # Create and train the network
    network = NeuralNetwork(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
    training_history = network.train(x_train, y_train, EPOCHS, LEARNING_RATE, BATCH_SIZE, validation_data=(x_val, y_val))

## 5. Results and Visualization

After training, we'll visualize the learning process and test the model's prediction on a random image from the test set.

In [ ]:
def plot_learning_curve(history, epochs):
    losses, accuracies, val_losses, val_accuracies = history
    
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(range(epochs), accuracies, label='Training Accuracy')
    plt.plot(range(epochs), val_accuracies, label='Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.title('Group 4: MNIST Learning Curve (Accuracy)')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(range(epochs), losses, label='Training Loss')
    plt.plot(range(epochs), val_losses, label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Cross-Entropy Loss')
    plt.title('Group 4: MNIST Learning Curve (Loss)')
    plt.legend()
    
    plt.tight_layout()
    plt.show()

In [ ]:
def predict_and_visualize(network, x_test, y_test_labels):
    # Select 3 random images from the test set
    indices = np.random.randint(0, x_test.shape[0], 3)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    for i, ax in enumerate(axes):
        index = indices[i]
        image = x_test[index]
        label = y_test_labels[index]
        
        # Reshape image for prediction
        prediction_hat = network.forward(image.reshape(1, -1))
        prediction = network.get_predictions(prediction_hat)[0]
        
        # Visualize the image and the prediction
        ax.imshow(image.reshape(28, 28), cmap='gray')
        ax.set_title(f"Actual: {label} | Predicted: {prediction}")
        ax.axis('off')
        
    plt.show()

In [ ]:
if 'train_df' in globals():
    plot_learning_curve(training_history, EPOCHS)

In [ ]:
if 'train_df' in globals():
    # Make a prediction on a random test image
    predict_and_visualize(network, x_test, y_test_labels)

### Confusion Matrix

The confusion matrix is a powerful tool to visualize the performance of a classification model. Each row represents the actual class, while each column represents the predicted class. It helps us understand which digits the model is confusing with others.

In [ ]:
def plot_confusion_matrix(network, x_test, y_test_labels):
    # Get predictions for the entire test set
    y_hat = network.forward(x_test)
    predictions = network.get_predictions(y_hat)
    
    # Create the confusion matrix
    cm = np.zeros((10, 10), dtype=int)
    for i in range(len(y_test_labels)):
        cm[y_test_labels[i], predictions[i]] += 1
        
    # Plot the confusion matrix
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(cm, cmap='viridis')
    
    # Add labels and title
    ax.set_xticks(np.arange(10))
    ax.set_yticks(np.arange(10))
    ax.set_xlabel('Predicted labels')
    ax.set_ylabel('True labels')
    ax.set_title('Confusion Matrix')
    
    # Add text annotations
    for i in range(10):
        for j in range(10):
            ax.text(j, i, cm[i, j], ha='center', va='center', color='white' if cm[i, j] > (cm.max() / 2) else 'black')
            
    plt.show()

In [ ]:
if 'train_df' in globals():
    plot_confusion_matrix(network, x_test, y_test_labels)

## 6. Conclusion

This project successfully implements a neural network from scratch using only NumPy to classify the MNIST handwritten digits. The model achieved **[TODO: insert final accuracy]%** accuracy on the test set.

A key feature of this implementation is the use of **vectorization** for the forward and backward propagation steps. Instead of iterating through each training example one by one, vectorization allows us to perform matrix operations on the entire dataset (or a mini-batch) at once. This leverages the highly optimized, low-level linear algebra libraries that NumPy is built on, resulting in a significant performance increase compared to an iterative approach. This is crucial for training models on large datasets.

Furthermore, the use of **He initialization** helps to ensure stable gradients during training, particularly when using the ReLU activation function, preventing issues like vanishing or exploding gradients and leading to more reliable convergence.

Analysis of the learning curve shows that the training and validation loss decrease together, indicating that the model is learning well without significant overfitting. If the validation loss were to start increasing while the training loss continued to decrease, it would be a sign of overfitting, and techniques like dropout or regularization could be introduced.

The confusion matrix reveals that the model performs well, with most predictions lying on the diagonal. Analysis of the confusion matrix shows the highest error rate between digits that are structurally similar, such as 4 and 9, due to their similar topology in low-resolution grayscale.

### Cybersecurity Connection: Adversarial Robustness

Understanding the raw weights and mathematical operations of a neural network, as we have done in this project, is the first step in understanding its vulnerabilities. In cybersecurity, **Adversarial Attacks** involve adding imperceptible noise to an input (like an image) to deliberately cause the model to make an incorrect prediction. For example, a carefully crafted noise pattern could make an image that looks like a '3' to a human be classified as an '8' by the AI. By building this network from scratch, we gain a deeper intuition into how these manipulations work and how we might build more robust models to defend against them.